In [ ]:
from backtesting import Backtest, Strategy
import pandas as pd
from backtesting.lib import crossover, plot_heatmaps, resample_apply
import seaborn as sns
import matplotlib.pyplot as plt
import mpld3
import numpy as np
import time
from talib import CDLENGULFING, ADX, CCI, ATR
from DataPaths import data_paths
import plotly.express as px

In [ ]:
class SimpleEngulfingStrategy(Strategy):
    SL = 50
    TP_R = 4
    days_held = 1
    MAX_HOLD_TIME = pd.Timedelta(days=days_held)

    def init(self):
        self.engulfing = self.I(CDLENGULFING, self.data.Open, self.data.High, self.data.Low, self.data.Close)
        self.entry_price = None
        self.stop_loss = None
        self.take_profit = None
        self.entry_time = None

    def next(self):
        engulf = self.engulfing[-1]
        price = self.data.Close[-1]
        current_time = self.data.index[-1]

        if current_time.weekday() in [0, 3]:
            return

        if not self.position:
            if engulf == 100:
                self.buy()
                self.entry_price = price
                self.stop_loss = price - self.SL
                self.take_profit = price + self.SL * self.TP_R
                self.entry_time = current_time

            elif engulf == -100:
                self.sell()
                self.entry_price = price
                self.stop_loss = price + self.SL
                self.take_profit = price - self.SL * self.TP_R
                self.entry_time = current_time

        if self.position:
            if self.position.is_long:
                if (
                    price <= self.stop_loss
                    or price >= self.take_profit
                    or (current_time - self.entry_time) > self.MAX_HOLD_TIME
                ):
                    self.position.close()
                    self.entry_price = None
                    self.stop_loss = None
                    self.take_profit = None
                    self.entry_time = None

            elif self.position.is_short:
                if (
                    price >= self.stop_loss
                    or price <= self.take_profit
                    or (current_time - self.entry_time) > self.MAX_HOLD_TIME
                ):
                    self.position.close()
                    self.entry_price = None
                    self.stop_loss = None
                    self.take_profit = None
                    self.entry_time = None


In [ ]:
def calculate_rows_per_timeframe(timeframe_minutes):
    """
    Calculate rows per day and rows per 30 days based on the given timeframe in minutes.
    """
    rows_per_day = (24 * 60) // timeframe_minutes  # Total minutes in a day divided by timeframe
    rows_per_30_days = 30 * rows_per_day  # 30 days of data
    return rows_per_day, rows_per_30_days

In [ ]:
def load_and_prepare_data(file_path, start_date, end_date):
    # Load CSV
    data = pd.read_csv(file_path, parse_dates=['Time'], index_col='Time')
    # Ensure index is datetime
    data.index = pd.to_datetime(data.index)

    # Print column names to verify
    print("Columns in the CSV file:", data.columns)

    # Select required columns
    data = data[['Open', 'High', 'Low', 'Close', 'Volume']].copy()

    # Remove duplicate indexes
    if data.index.duplicated().any():
        print("Duplicate indexes found. Removing duplicates.")
        data = data[~data.index.duplicated(keep='first')]

    # Check for NaN values
    print("Checking for NaN values in the data:")
    print(data.isna().sum())

    # Drop NaN values
    data = data.dropna()

    # Reduce the number of data points to a specific date range
    data = data.loc[start_date:end_date]

    print(f"Number of data points after reduction: {len(data)}")

    return data


In [ ]:
if __name__ == '__main__':
    file_path = data_paths["GBPUSD"]["M15"]
    
    start_date = '2022-01-01'
    end_date = '2025-01-01'
    
    timeframe_minutes = 15  # Change this to your desired timeframe
    rows_per_day, rows_per_30_days = calculate_rows_per_timeframe(timeframe_minutes)
    
    df = load_and_prepare_data(file_path, start_date, end_date)
    
    returns = []
    
    # Process data slices for backtesting
    for x in range(rows_per_30_days, len(df) + 1, rows_per_30_days):
        bt = Backtest(df.iloc[x-rows_per_30_days:x], 
                      SimpleEngulfingStrategy, cash=10_000_000, commission=.002)
        stats = bt.run()

        if "Return [%]" in stats:
            returns.append(stats["Return [%]"])
    
    # Plot the results
    fig = px.box(returns, points="all")
    fig.update_layout(
        xaxis_title="Strategy",
        yaxis_title="Returns (%)",
    )
    fig.show()